In [1]:
import os
import sys

from dotenv import load_dotenv

root_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

print(f"Root directory: {root_dir}")
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

load_dotenv(os.path.join(root_dir, ".env"))

Root directory: /home/victor-wsl/programming/curso-ia


True

In [2]:
from pathlib import Path

from src.mfe.config.logging import create_logger

logger, error_logger = create_logger(name="docling_pipeline")

In [ ]:
from huggingface_hub import CommitOperationAdd, CommitOperationDelete, HfApi

output_jsonl_final_dir = os.path.join(
    root_dir, "data", "output", "docling_parallel", "jsonl_final"
)

HF_TOKEN = os.getenv("HF_TOKEN")  # token em .env com permissão write
HF_USERNAME = os.getenv("HF_USERNAME")  # seu username no HF
DATASET_NAME = "ia-course"  # nome do repositório de dataset
DATASET_VERSION = "0.0.2"
EXTRACTION_MODEL = "docling"

# repo_id = f"{HF_USERNAME}/{DATASET_NAME}"
repo_id = f"Cursos-IA/{DATASET_NAME}"

# ---------------------------------------------------------------------------
# Dataset card (README.md)
# ---------------------------------------------------------------------------
dataset_card = f"""\
---
license: other
tags:
  - ocr
  - pdf
  - text-extraction
  - {EXTRACTION_MODEL}
version: "{DATASET_VERSION}"
configs:
  - config_name: default
    data_files:
      - split: train
        path: "data/*.jsonl"
---

# {DATASET_NAME}

Extracted text dataset generated from PDF and document files using OCR. This version has been processed with the {EXTRACTION_MODEL} model, and was merged with the metadata available in the mongodb.
In this version was added files inside of zip files.

## Extraction details

| Field | Value |
|---|---|
| Extraction model | `{EXTRACTION_MODEL}` |
| Dataset version | `{DATASET_VERSION}` |
| Pipeline | {EXTRACTION_MODEL} Pipeline |

## Schema

Each record contains:

| Field | Description |
|---|---|
| `id` | SHA-256 hash of file content + relative path |
| `file_format` | File extension (`.pdf`, `.docx`, etc.) |
| `tokens_count` | Token count (GPT-4o tokenizer) |
| `images_s3_uri` | S3 URI for extracted page images |
| `text_content` | Full extracted text content |
| `class_metadata` | Metadata about the class (turma/plano_aula) if matched |


## Versions

| Version | Tag | Notes |
|---|---|---|
| 0.0.2 | `v0.0.2` | First extraction using `{EXTRACTION_MODEL}` |
"""

api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=repo_id,
    repo_type="dataset",
    exist_ok=True,
    private=True,
)
logger.info(f"Repositório pronto: https://huggingface.co/datasets/{repo_id}")

# ---------------------------------------------------------------------------
# Delete existing data files from previous version
# ---------------------------------------------------------------------------
existing_files = [
    f.rfilename
    for f in api.list_repo_tree(repo_id=repo_id, repo_type="dataset", recursive=True)
    if hasattr(f, "rfilename") and f.rfilename.startswith("data/")
]
logger.info(f"Arquivos existentes para deletar: {len(existing_files)}")

operations = [CommitOperationDelete(path_in_repo=path) for path in existing_files]

# ---------------------------------------------------------------------------
# Add new JSONL files + README
# ---------------------------------------------------------------------------
jsonl_final_dir = Path(output_jsonl_final_dir)
operations += [
    CommitOperationAdd(
        path_in_repo=f"data/{jsonl_file.name}",
        path_or_fileobj=str(jsonl_file),
    )
    for jsonl_file in sorted(jsonl_final_dir.glob("*.jsonl"))
]

operations.append(
    CommitOperationAdd(
        path_in_repo="README.md",
        path_or_fileobj=dataset_card.encode("utf-8"),
    )
)

logger.info(
    f"Fazendo upload de {len([o for o in operations if isinstance(o, CommitOperationAdd)]) - 1} arquivos JSONL + README ({len(existing_files)} arquivos antigos deletados)..."
)

commit_info = api.create_commit(
    repo_id=repo_id,
    repo_type="dataset",
    revision=f"v.{DATASET_VERSION}-{EXTRACTION_MODEL}",
    operations=operations,
    commit_message=f"feat: add extraction v{DATASET_VERSION} using {EXTRACTION_MODEL}",
)
logger.info(f"Commit: {commit_info.commit_url}")

# Cria tag git para marcar a versão imutável
api.create_tag(
    repo_id=repo_id,
    repo_type="dataset",
    tag=f"v{DATASET_VERSION}",
    tag_message=f"First extraction using {EXTRACTION_MODEL}",
    revision=commit_info.oid,
)
logger.info(
    f"Tag v{DATASET_VERSION} criada → https://huggingface.co/datasets/{repo_id}/tree/v{DATASET_VERSION}"
)


/home/victor-wsl/programming/curso-ia/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-23 12:13:12,348 INFO:docling_pipeline:Repositório pronto: https://huggingface.co/datasets/Cursos-IA/ia-course
2026-03-23 12:13:13,032 INFO:docling_pipeline:Arquivos existentes para deletar: 8
2026-03-23 12:13:13,166 INFO:docling_pipeline:Fazendo upload de 8 arquivos JSONL + README (8 arquivos antigos deletados)...
Processing Files (7 / 7): 100%|██████████| 80.1MB / 80.1MB, 14.8MB/s  
New Data Upload: 100%|██████████| 21.6MB / 21.6MB, 4.01MB/s  
2026-03-23 12:13:23,150 INFO:docling_pipeline:Commit: https://huggingface.co/datasets/Cursos-IA/ia-course/commit/ef2ba1e36ef3085229d82d89f0551429da477e8b
2026-03-23 12:13:23,632 INFO:docling_pipeline:Tag v0.0.2 criada → https://huggingface.co/datasets/Cursos-IA/ia-c